# MaskWall: retrain ONLY arm B (seed 42) and save to Google Drive as it goes

Why: the first Colab run died, so its checkpoints are lost. Arm B (negatives + matched attacks) passed the four pre-registered criteria in that run
(see `docs/decisions.md`, Day 2.9). This notebook re-creates **one** checkpoint, same data, same code, same hyperparameters, seed 42.
It will not be bit-identical to the first run (GPU fp16 is not fully deterministic); the last cell prints the new numbers next to the original ones.
After you download the checkpoint, **int8 export and the re-evaluation happen on your PC**, and the served model is only replaced if it still passes there.

**Everything that matters is kept on Drive** (`MyDrive/maskwall_armB_seed42/`): the upload zip, a resumable state after every epoch, the final checkpoint.
If Colab disconnects, reopen the notebook and **Run all again**: it picks up from the last finished epoch and skips what is done.
Training is about 4 epochs x 1 minute on a T4 (an estimate, not a measurement).

## Before you run
1. **Runtime -> Change runtime type -> T4 GPU.**
2. Have `maskwall_hardneg_upload.zip` ready (in `colab_upload/` in the project). You only upload it **once**: it is copied to Drive.
3. **Run all.** Allow the Drive pop-up when it asks. Do not change anything after seeing numbers.

## 1. Setup

In [ ]:
!pip -q install "transformers>=4.44" accelerate sentencepiece protobuf scikit-learn pandas pyarrow joblib tabulate

In [ ]:
import hashlib, json, os, platform, random, re, sys, time, zipfile
import numpy as np
import pandas as pd
import torch

DRY_RUN = os.environ.get("MW_DRY_RUN") == "1"          # local CPU smoke test only; always False in Colab
DEV = "cuda" if torch.cuda.is_available() else "cpu"
assert DEV == "cuda" or DRY_RUN, "No GPU: Runtime -> Change runtime type -> T4 GPU"
print(torch.__version__, torch.cuda.get_device_name(0) if DEV == "cuda" else "CPU (dry run)")

# ------------------------------------------------------------------ config (fixed before any result)
SEEDS = [42]
HF_MODEL, LR, BATCH = "sentence-transformers/all-MiniLM-L6-v2", 5e-5, 32
HEAD, TAIL = 256, 254
MAX_EPOCHS, MIN_EPOCHS, PATIENCE = 4, 3, 1
WARMUP_FRAC, WEIGHT_DECAY, GRAD_CLIP = 0.1, 0.01, 1.0
FP16 = DEV == "cuda"
CONTENT, STRIDE, MAX_WINDOWS = 510, 255, 6              # same as backend/app/scanners/windowing.py and the served default (sliding, cap 6)
ARMS = {"B_neg_plus_attacks": ["T1", "T2", "T3"]}
RUN_ORIGINAL_LABELS_CHECK = False
USE_DRIVE = True                                          # always: this is the point of this notebook

ROOT = os.environ.get("MW_ROOT", "/content/maskwall")
OUT = os.environ.get("MW_OUT", "/content/out")
if USE_DRIVE and not DRY_RUN:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/maskwall_armB_seed42"
os.makedirs(ROOT, exist_ok=True); os.makedirs(OUT, exist_ok=True)

if DRY_RUN:                                             # tiny and fast
    SEEDS, MAX_EPOCHS, MIN_EPOCHS, BATCH = [42], int(os.environ.get('MW_DRY_EPOCHS', '1')), 1, 8


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if DEV == "cuda":
        torch.cuda.manual_seed_all(s)
REF = {"test_f1": 0.909, "block": 0.948, "E1_fp": 2, "E2_fp": 1, "E3_caught": 23, "test_recall": 0.889, "indirect_recall": 0.927, "tricky_attack_recall": 0.81, "best_epoch": 4}                    # the ORIGINAL arm B seed 42 numbers (hardneg_results.json), for the comparison at the end

## 2. Upload the zip (once; cached on Drive) and load the data

In [ ]:
ZIP_ON_DRIVE = f"{OUT}/maskwall_hardneg_upload.zip"
if not DRY_RUN:
    if not os.path.exists(ZIP_ON_DRIVE):
        from google.colab import files
        up = files.upload()                                  # select maskwall_hardneg_upload.zip
        assert "maskwall_hardneg_upload.zip" in up, f"expected maskwall_hardneg_upload.zip, got {list(up)}"
        open(ZIP_ON_DRIVE, "wb").write(up["maskwall_hardneg_upload.zip"])
        print("saved the zip to Drive; after a disconnect you will not need to upload it again")
    else:
        print("using the zip already on Drive")
    zipfile.ZipFile(ZIP_ON_DRIVE).extractall(ROOT)
sys.path.insert(0, ROOT)
import train_baseline as tb                              # the SAME evaluation code as every earlier result
print(sorted(os.listdir(ROOT)), sorted(os.listdir(f"{ROOT}/data/processed")))

In [ ]:
P = lambda n: pd.read_parquet(f"{ROOT}/data/processed/{n}.parquet")
S = {n: P(n) for n in ["train_augmented", "val", "test", "xstest_safe", "indirect_test", "long_eval",
                       "hinglish_human_benign", "hinglish_attacks_ai", "prompt_talk_benign_ai", "manual7"]}
tricky = pd.read_csv(f"{ROOT}/data/tricky.csv")
hn_tr = pd.read_csv(f"{ROOT}/data/hardneg/hardneg_train.csv")
hn_va = pd.read_csv(f"{ROOT}/data/hardneg/hardneg_val.csv")

for n, d in S.items():
    print(f"{n:24s} {len(d):5d} rows" + (f", {int(d.label.sum())} attacks" if "label" in d else ""))
print(f"{'tricky':24s} {len(tricky):5d} rows")
print("hardneg train:", hn_tr.groupby(["tier", "label"]).size().to_dict())
print("hardneg val:  ", hn_va.groupby(["tier", "label"]).size().to_dict())

# ---- leakage: exact-text overlap between the NEW training/val rows and every evaluation set (build_hardneg.py also checked near-duplicates)
norm = lambda t: " ".join(t.split()).lower()
eval_text = set()
for n in ["val", "test", "xstest_safe", "indirect_test", "long_eval", "hinglish_human_benign", "hinglish_attacks_ai", "prompt_talk_benign_ai", "manual7"]:
    eval_text |= set(map(norm, S[n].text))
eval_text |= set(map(norm, S["hinglish_human_benign"].en_control)) | set(map(norm, tricky.text))
assert not set(map(norm, hn_tr.text)) & eval_text, "hardneg train overlaps an evaluation set"
assert not set(map(norm, hn_va.text)) & eval_text, "hardneg val overlaps an evaluation set"
assert not set(map(norm, hn_tr.text)) & set(map(norm, hn_va.text)), "hardneg train overlaps hardneg val"
assert not set(map(norm, S["train_augmented"].text)) & set(map(norm, S["val"].text)) and not set(map(norm, S["train_augmented"].text)) & set(map(norm, S["test"].text))
print("no exact overlap between training data and any evaluation/validation set")

if DRY_RUN:
    S["train_augmented"] = S["train_augmented"][S["train_augmented"].text.str.len() < 600].sample(300, random_state=0).reset_index(drop=True)   # short rows only: CPU memory
    S["val"] = S["val"].sample(150, random_state=0).reset_index(drop=True)
    hn_tr = hn_tr.groupby("tier").head(60).reset_index(drop=True)
    hn_va = hn_va.groupby("tier").head(20).reset_index(drop=True)
    for _n, _k in [("test", 100), ("xstest_safe", 40), ("indirect_test", 30), ("long_eval", 8)]:      # keep the CPU smoke test fast and small in memory
        S[_n] = S[_n].sample(_k, random_state=0).reset_index(drop=True)

## 3. Tokenisation, prediction, and a training loop that saves to Drive after every epoch

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.metrics import average_precision_score, roc_auc_score


def encode_head_tail(tok, texts):
    ids_all = tok(list(texts), add_special_tokens=False, truncation=False)["input_ids"]
    out = []
    for ids in ids_all:
        if len(ids) > HEAD + TAIL:
            ids = ids[:HEAD] + ids[-TAIL:]
        out.append([tok.cls_token_id] + ids + [tok.sep_token_id])
    return out


def window_spans(n, content=CONTENT, stride=STRIDE):                 # copy of backend/app/scanners/windowing.py
    if n <= content:
        return [(0, n)]
    spans, s = [], 0
    while s + content < n:
        spans.append((s, s + content)); s += stride
    spans.append((n - content, n))
    return spans


def select_windows(spans, cap):
    if len(spans) <= cap:
        return spans, False
    return spans[:cap - 1] + [spans[-1]], True


def collate(seqs, pad_id):
    L = max(len(s) for s in seqs)
    ids = torch.full((len(seqs), L), pad_id, dtype=torch.long)
    att = torch.zeros((len(seqs), L), dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, :len(s)] = torch.tensor(s); att[i, :len(s)] = 1
    return ids, att


@torch.no_grad()
def predict_enc(model, enc, pad_id, bs=64 if DEV == 'cuda' else 8):
    model.eval()
    order = np.argsort([len(x) for x in enc])
    probs = np.zeros(len(enc))
    for i in range(0, len(enc), bs):
        idx = order[i:i + bs]
        ids, att = collate([enc[j] for j in idx], pad_id)
        with torch.autocast(DEV, dtype=torch.float16, enabled=FP16):
            logits = model(input_ids=ids.to(DEV), attention_mask=att.to(DEV)).logits
        probs[idx] = torch.softmax(logits.float(), -1)[:, 1].cpu().numpy()
    return probs


def predict(model, tok, texts):
    return predict_enc(model, encode_head_tail(tok, texts), tok.pad_token_id)


def predict_sliding(model, tok, texts):
    ids_all = tok(list(texts), add_special_tokens=False, truncation=False)["input_ids"]
    flat, owner = [], []
    for d, ids in enumerate(ids_all):
        spans, _ = select_windows(window_spans(len(ids)), MAX_WINDOWS)
        for s, e in spans:
            flat.append([tok.cls_token_id] + ids[s:e] + [tok.sep_token_id]); owner.append(d)
    p = predict_enc(model, flat, tok.pad_token_id, bs=32 if DEV == 'cuda' else 4)
    out = np.zeros(len(ids_all))
    for pi, d in zip(p, owner):
        out[d] = max(out[d], pi)
    return out

In [ ]:
def _atomic_save(obj, path):
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)                                   # a disconnect during the write never leaves a half-written file


def train_one(tag, seed, train_df, val_df, state_dir):
    # Same recipe as train_hardneg.ipynb (AdamW, linear warmup, fp16, best epoch by val AP, early stop from epoch 3), plus a resumable state on Drive.
    os.makedirs(state_dir, exist_ok=True)
    STATE, BEST = f"{state_dir}/state.pt", f"{state_dir}/best.pt"
    set_seed(seed)
    tok = AutoTokenizer.from_pretrained(HF_MODEL); tok.model_max_length = 10**6
    model = AutoModelForSequenceClassification.from_pretrained(HF_MODEL, num_labels=2).float().to(DEV)
    enc_tr, enc_va = encode_head_tail(tok, train_df.text), encode_head_tail(tok, val_df.text)
    y_tr, y_va = train_df.label.to_numpy(), val_df.label.to_numpy()
    n_tr = len(enc_tr)
    total = int(np.ceil(n_tr / BATCH)) * MAX_EPOCHS
    no_decay = ["bias", "LayerNorm.weight", "layernorm", "LayerNorm.bias"]
    params = [{"params": [p for n, p in model.named_parameters() if not any(k in n for k in no_decay)], "weight_decay": WEIGHT_DECAY},
              {"params": [p for n, p in model.named_parameters() if any(k in n for k in no_decay)], "weight_decay": 0.0}]
    opt = torch.optim.AdamW(params, lr=LR)
    sched = get_linear_schedule_with_warmup(opt, int(WARMUP_FRAC * total), total)
    scaler = torch.amp.GradScaler("cuda", enabled=FP16)
    history, best_ap, best_epoch, stale, done_epochs, finished, secs0 = [], -1.0, 0, 0, 0, False, 0.0
    if os.path.exists(STATE):                                # ---- resume
        st = torch.load(STATE, map_location="cpu", weights_only=False)
        model.load_state_dict(st["model"]); opt.load_state_dict(st["opt"]); sched.load_state_dict(st["sched"]); scaler.load_state_dict(st["scaler"])
        history, best_ap, best_epoch, stale, done_epochs, finished, secs0 = st["history"], st["best_ap"], st["best_epoch"], st["stale"], st["epoch"], st["finished"], st["secs"]
        print(f"[{tag}] RESUMING after epoch {done_epochs} (finished={finished}); history so far: {[(h['epoch'], round(h['val_ap'], 4)) for h in history]}")
    t0 = time.time()
    for epoch in range(done_epochs + 1, MAX_EPOCHS + 1):
        if finished:
            break
        model.train()
        order = np.random.default_rng(seed + epoch).permutation(n_tr)
        running = 0.0
        for i in range(0, n_tr, BATCH):
            idx = order[i:i + BATCH]
            ids, att = collate([enc_tr[j] for j in idx], tok.pad_token_id)
            labels = torch.tensor(y_tr[idx], dtype=torch.long).to(DEV)
            with torch.autocast(DEV, dtype=torch.float16, enabled=FP16):
                loss = model(input_ids=ids.to(DEV), attention_mask=att.to(DEV), labels=labels).loss
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(opt); scaler.update(); sched.step()
            running += loss.item() * len(idx)
        vs = predict_enc(model, enc_va, tok.pad_token_id)
        ap = float(average_precision_score(y_va, vs))
        history.append({"epoch": epoch, "train_loss": running / n_tr, "val_ap": ap, "val_best_f1": tb.tune_f1_threshold(y_va, vs)[1]})
        print(f"[{tag}] epoch {epoch}: loss {running / n_tr:.4f}  val AP {ap:.4f}  ({secs0 + time.time() - t0:.0f}s)")
        if ap > best_ap:
            best_ap, best_epoch, stale = ap, epoch, 0
            _atomic_save({k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, BEST)
        else:
            stale += 1
            if epoch >= MIN_EPOCHS and stale >= PATIENCE:
                print(f"[{tag}] early stop after epoch {epoch} (best {best_epoch})")
                finished = True
        finished = finished or epoch == MAX_EPOCHS
        _atomic_save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(), "scaler": scaler.state_dict(), "history": history,
                      "best_ap": best_ap, "best_epoch": best_epoch, "stale": stale, "epoch": epoch, "finished": finished, "secs": secs0 + time.time() - t0}, STATE)
        print(f"[{tag}] state saved to Drive after epoch {epoch}")
    model.load_state_dict(torch.load(BEST, map_location="cpu"))
    return model, tok, history, best_epoch, secs0 + time.time() - t0

## 4. Evaluation (copied unchanged from the main notebook)

In [ ]:
def rate(mask):
    mask = np.asarray(mask, bool)
    return {"k": int(mask.sum()), "n": int(len(mask))}


def thresholds(y, scores):
    t_f1, f1 = tb.tune_f1_threshold(y, scores)
    t_fpr = tb.tune_fpr_threshold(y, scores)
    return {"thr_f1": t_f1, "thr_fpr1pct": t_fpr, "block": max(t_f1, t_fpr), "warn": min(t_f1, t_fpr), "val_f1": f1}


def sets_for_scoring():
    # name -> list of texts, in a fixed order (the saved raw scores are positional)
    h = S["hinglish_human_benign"]
    return {"test": S["test"].text, "tricky": tricky.text, "xstest_safe": S["xstest_safe"].text, "indirect_test": S["indirect_test"].text,
            "E1_hinglish": h.text, "E1_english_control": h.en_control, "E2_prompt_talk": S["prompt_talk_benign_ai"].text,
            "E3_hinglish_attacks": S["hinglish_attacks_ai"].text, "E4_manual7": S["manual7"].text}


def evaluate(model, tok, thr, label_col="label"):
    # Score every evaluation set with the frozen thresholds in `thr` (from thresholds()). Returns (metrics, raw scores).
    sc = {n: predict(model, tok, list(t)) for n, t in sets_for_scoring().items()}
    sc["long_eval"] = predict_sliding(model, tok, list(S["long_eval"].text))
    blk, wrn = thr["block"], thr["warn"]
    test_y = S["test"][label_col].to_numpy()
    m = {"thresholds": thr}
    m["test"] = {**tb.rates(test_y, sc["test"] >= blk), "roc_auc": float(roc_auc_score(test_y, sc["test"])), "pr_auc": float(average_precision_score(test_y, sc["test"])),
                 "at_f1_threshold": tb.rates(test_y, sc["test"] >= thr["thr_f1"])}
    t = S["test"].assign(label=test_y)
    m["test_full_eval_at_block"] = tb.evaluate_split(t, sc["test"], blk, thr["thr_fpr1pct"])
    tk = tricky.assign(pred=sc["tricky"] >= blk)
    att = tk[tk.label == 1]
    m["tricky"] = {"overall": int((tk.pred == tk.label).sum()), "n": len(tk), "attack_recall": rate(att.pred), "benign_fp": rate(tk[tk.label == 0].pred),
                   "by_category": {c: {"correct": int((g.pred == g.label).sum()), "n": len(g), "label": int(g.label.iloc[0])} for c, g in tk.groupby("category")}}
    m["xstest_safe_fp"] = rate(sc["xstest_safe"] >= blk)
    ie = S["indirect_test"]
    m["indirect_test"] = {"recall": rate((sc["indirect_test"] >= blk)[ie.label.to_numpy() == 1]), "fpr": rate((sc["indirect_test"] >= blk)[ie.label.to_numpy() == 0])}
    m["E1"] = {"fp_block": rate(sc["E1_hinglish"] >= blk), "fp_warn": rate(sc["E1_hinglish"] >= wrn), "english_control_fp_block": rate(sc["E1_english_control"] >= blk)}
    p = S["prompt_talk_benign_ai"]
    m["E2"] = {"fp_block": rate(sc["E2_prompt_talk"] >= blk), "fp_warn": rate(sc["E2_prompt_talk"] >= wrn),
               "by_language": {lg: rate((sc["E2_prompt_talk"] >= blk)[(p.language == lg).to_numpy()]) for lg in p.language.unique()},
               "by_category": {c: rate((sc["E2_prompt_talk"] >= blk)[(p.category == c).to_numpy()]) for c in p.category.unique()}}
    a = S["hinglish_attacks_ai"]
    m["E3"] = {"caught_block": rate(sc["E3_hinglish_attacks"] >= blk), "caught_warn": rate(sc["E3_hinglish_attacks"] >= wrn),
               "by_category": {c: rate((sc["E3_hinglish_attacks"] >= blk)[(a.category == c).to_numpy()]) for c in a.category.unique()}}
    m["E4"] = [{"text": x[:60], "label": int(l), "score": float(s)} for x, l, s in zip(S["manual7"].text, S["manual7"].label, sc["E4_manual7"])]
    lg = S["long_eval"]
    s_long = sc["long_eval"]
    m["long_eval_sliding"] = {"benign_fpr_block": rate((s_long >= blk)[lg.label.to_numpy() == 0]),
                              "recall_by_position": {pos: rate((s_long >= blk)[((lg.label == 1) & (lg.position == pos)).to_numpy()]) for pos in ["start", "middle", "end"]}}
    return m, sc

## 5. Train arm B, seed 42 (resumable), tune thresholds on its own val, evaluate, save the final checkpoint to Drive
Thresholds come from the arm's val only (old val + the new val slice of T1/T2/T3), exactly as in the main round: block = max(thr_F1, thr_FPR<=1%), warn = min.

In [ ]:
ARM, SEED = "B_neg_plus_attacks", 42
tiers = ARMS[ARM]
tr = pd.concat([S["train_augmented"][["text", "label"]], hn_tr[hn_tr.tier.isin(tiers)][["text", "label"]]], ignore_index=True)
va = pd.concat([S["val"][["text", "label"]], hn_va[hn_va.tier.isin(tiers)][["text", "label"]]], ignore_index=True)
print(f"{ARM} seed {SEED}: {len(tr)} train rows ({int(tr.label.sum())} attacks), {len(va)} val rows")

FINAL = f"{OUT}/final_armB_seed42"
model, tok, history, best_epoch, secs = train_one(f"{ARM}/seed{SEED}", SEED, tr, va, f"{OUT}/state_armB_seed42")
vs = predict(model, tok, list(va.text))
thr = thresholds(va.label.to_numpy(), vs)
m, sc = evaluate(model, tok, thr)
m.update({"arm": ARM, "seed": SEED, "history": history, "best_epoch": best_epoch, "train_seconds": round(secs), "n_train": len(tr), "n_val": len(va)})
os.makedirs(FINAL, exist_ok=True)
model.save_pretrained(FINAL); tok.save_pretrained(FINAL)                 # config.json, model.safetensors, tokenizer files
json.dump({"result": m, "scores": {k: [round(float(x), 5) for x in v] for k, v in sc.items()}, "ref_original_run": REF}, open(f"{FINAL}/colab_eval.json", "w"), default=float)
print("saved the final checkpoint to", FINAL, sorted(os.listdir(FINAL)))

## 6. Does it look like the original arm B seed 42? (a sanity check, not a decision)

In [ ]:
f = m["test"]
now = {"test_f1": round(f["f1"], 3), "block": round(thr["block"], 3), "E1_fp": m["E1"]["fp_block"]["k"], "E2_fp": m["E2"]["fp_block"]["k"], "E3_caught": m["E3"]["caught_block"]["k"],
       "test_recall": round(f["recall"], 3), "indirect_recall": round(m["indirect_test"]["recall"]["k"] / m["indirect_test"]["recall"]["n"], 3),
       "tricky_attack_recall": round(m["tricky"]["attack_recall"]["k"] / m["tricky"]["attack_recall"]["n"], 3), "best_epoch": best_epoch}
print(f"{'metric':22s} {'original run':>14s} {'this retrain':>14s}")
for k in REF:
    print(f"{k:22s} {REF[k]:>14} {now[k]:>14}")
print("\nExpect small differences (fp16 GPU nondeterminism; the first run's seed spread was E1 1-2, E2 1-2, E3 22-24 of 24). Do NOT retrain to get nicer numbers.")

## 7. Download the checkpoint (it also stays on Drive)

In [ ]:
import shutil
if not DRY_RUN:
    from google.colab import files
    shutil.make_archive("/content/hardneg_B_seed42", "zip", FINAL)
    files.download("/content/hardneg_B_seed42.zip")
    print("Downloaded hardneg_B_seed42.zip. Put it in models/ckpt/ on your PC (the zip itself is fine; the export script unpacks it).")